In [1]:
# APPROACH 1: TVAR ROUTING SIMULATION
# Demonstrates routing improvement without modifying NS-3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("="*60)
print("TVAR-GUIDED ROUTING SIMULATOR")
print("="*60)

# Load NS-3 data
df = pd.read_csv('mesh_packet_loss_data.csv')

# Prepare routing simulation
def simulate_tvar_routing(df, trained_models, optimal_order):
    """
    Simulate TVAR-guided routing decisions
    
    Returns: tvar_results, aodv_results (DataFrames)
    """
    
    # Get unique time windows
    windows = sorted(df['time_window'].unique())
    
    tvar_decisions = []
    aodv_decisions = []
    
    for i in range(optimal_order, len(windows) - 1):
        current_window = windows[i]
        next_window = windows[i + 1]
        
        # Get data for current window
        current_data = df[df['time_window'] == current_window]
        next_data = df[df['time_window'] == next_window]
        
        # TVAR: Predict losses for all routes at next window
        tvar_predictions = {}
        
        for route_id, model_data in trained_models.items():
            source_node = int(route_id.split('_')[1])
            
            # Historical data up to current window
            route_history = df[(df['source_node'] == source_node) & 
                            (df['time_window'] <= current_window)]['loss_rate'].values
            
            if len(route_history) < optimal_order:
                tvar_predictions[source_node] = 0.5  # Default
                continue
            
            # Make prediction
            try:
                model = model_data['model']
                y_pred, _ = model.predict_one_step(route_history, len(route_history)-1)
                tvar_predictions[source_node] = np.clip(y_pred, 0, 1)
            except:
                tvar_predictions[source_node] = np.mean(route_history[-5:])
        
        # TVAR chooses route with LOWEST predicted loss
        tvar_chosen_node = min(tvar_predictions.keys(), key=lambda k: tvar_predictions[k])
        tvar_predicted_loss = tvar_predictions[tvar_chosen_node]
        
        # What actually happened on TVAR's chosen route?
        tvar_actual_data = next_data[next_data['source_node'] == tvar_chosen_node]
        if len(tvar_actual_data) > 0:
            tvar_actual_loss = tvar_actual_data['loss_rate'].values[0]
        else:
            tvar_actual_loss = 0.5
        
        # AODV: Reactive - chooses based on PAST performance (last 3 windows)
        recent_data = df[df['time_window'].between(
            max(0, current_window - 3), 
            current_window
        )]
        avg_past_losses = recent_data.groupby('source_node')['loss_rate'].mean()
        
        if len(avg_past_losses) > 0:
            aodv_chosen_node = avg_past_losses.idxmin()
        else:
            aodv_chosen_node = current_data['source_node'].iloc[0]
        
        # What actually happened on AODV's chosen route?
        aodv_actual_data = next_data[next_data['source_node'] == aodv_chosen_node]
        if len(aodv_actual_data) > 0:
            aodv_actual_loss = aodv_actual_data['loss_rate'].values[0]
        else:
            aodv_actual_loss = 0.5
        
        # Record decisions
        tvar_decisions.append({
            'window': next_window,
            'chosen_node': tvar_chosen_node,
            'predicted_loss': tvar_predicted_loss,
            'actual_loss': tvar_actual_loss
        })
        
        aodv_decisions.append({
            'window': next_window,
            'chosen_node': aodv_chosen_node,
            'actual_loss': aodv_actual_loss
        })
    
    return pd.DataFrame(tvar_decisions), pd.DataFrame(aodv_decisions)

# Run simulation
tvar_results, aodv_results = simulate_tvar_routing(df, trained_models, optimal_order)

print(f"\n✓ Simulated {len(tvar_results)} routing decisions")
print(f"✓ Time windows analyzed: {tvar_results['window'].min()} to {tvar_results['window'].max()}")

TVAR-GUIDED ROUTING SIMULATOR


NameError: name 'trained_models' is not defined

In [ ]:
# Calculate PDR improvement

print("\n" + "="*60)
print("ROUTING PERFORMANCE ANALYSIS")
print("="*60)

# Average packet loss
tvar_avg_loss = tvar_results['actual_loss'].mean()
aodv_avg_loss = aodv_results['actual_loss'].mean()

# Packet Delivery Rate (PDR) = 1 - loss_rate
tvar_pdr = (1 - tvar_avg_loss) * 100
aodv_pdr = (1 - aodv_avg_loss) * 100

# Improvement
pdr_improvement = tvar_pdr - aodv_pdr
relative_improvement = (pdr_improvement / aodv_pdr) * 100

print("\n📊 TVAR-Guided Routing:")
print(f"   Average Loss Rate: {tvar_avg_loss:.4f}")
print(f"   Packet Delivery Rate: {tvar_pdr:.2f}%")

print("\n📊 AODV Reactive Routing:")
print(f"   Average Loss Rate: {aodv_avg_loss:.4f}")
print(f"   Packet Delivery Rate: {aodv_pdr:.2f}%")

print("\n" + "="*60)
print(f"🎯 IMPROVEMENT: +{pdr_improvement:.2f}% PDR ({relative_improvement:+.1f}% relative)")
print("="*60)

# Additional metrics
print("\n📈 Detailed Analysis:")
print(f"   TVAR prediction accuracy (RMSE): {np.sqrt(np.mean((tvar_results['predicted_loss'] - tvar_results['actual_loss'])**2)):.4f}")
print(f"   Times TVAR chose better route than AODV: {(tvar_results['actual_loss'] < aodv_results['actual_loss']).sum()}/{len(tvar_results)}")
print(f"   Max loss avoided in single window: {max(0, (aodv_results['actual_loss'] - tvar_results['actual_loss']).max()):.4f}")

# Route diversity
print(f"\n🔀 Route Selection Diversity:")
print(f"   TVAR used {tvar_results['chosen_node'].nunique()} different routes")
print(f"   AODV used {aodv_results['chosen_node'].nunique()} different routes")
print(f"   Route switches (TVAR): {(tvar_results['chosen_node'].diff() != 0).sum()}")

In [ ]:
# Create publication-quality routing comparison plot

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Actual loss rates over time
axes[0].plot(tvar_results['window'], tvar_results['actual_loss'], 'o-', 
             label='TVAR-Guided Routing', color='steelblue', linewidth=2, 
             markersize=5, alpha=0.8)
axes[0].plot(aodv_results['window'], aodv_results['actual_loss'], 's-', 
             label='AODV Reactive Routing', color='coral', linewidth=2, 
             markersize=5, alpha=0.7)
axes[0].axhline(tvar_avg_loss, color='steelblue', linestyle='--', linewidth=1.5,
                alpha=0.6, label=f'TVAR Mean: {tvar_avg_loss:.3f}')
axes[0].axhline(aodv_avg_loss, color='coral', linestyle='--', linewidth=1.5,
                alpha=0.6, label=f'AODV Mean: {aodv_avg_loss:.3f}')
axes[0].fill_between(tvar_results['window'], tvar_results['actual_loss'], 
                      aodv_results['actual_loss'],
                      where=(tvar_results['actual_loss'] < aodv_results['actual_loss']),
                      color='green', alpha=0.2, label='TVAR Advantage')
axes[0].set_xlabel('Time Window', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Packet Loss Rate', fontsize=11, fontweight='bold')
axes[0].set_title('Routing Performance Comparison: Packet Loss Over Time', 
                  fontsize=13, fontweight='bold')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Cumulative PDR
tvar_cumulative_delivered = (1 - tvar_results['actual_loss']).cumsum()
aodv_cumulative_delivered = (1 - aodv_results['actual_loss']).cumsum()
windows_range = np.arange(1, len(tvar_results) + 1)

tvar_cumulative_pdr = (tvar_cumulative_delivered / windows_range) * 100
aodv_cumulative_pdr = (aodv_cumulative_delivered / windows_range) * 100

axes[1].plot(tvar_results['window'], tvar_cumulative_pdr, linewidth=2.5, 
             color='steelblue', label='TVAR-Guided')
axes[1].plot(aodv_results['window'], aodv_cumulative_pdr, linewidth=2.5, 
             color='coral', label='AODV Reactive')
axes[1].fill_between(tvar_results['window'], aodv_cumulative_pdr, tvar_cumulative_pdr,
                      where=(tvar_cumulative_pdr >= aodv_cumulative_pdr),
                      color='green', alpha=0.3, label='Cumulative Gain')
axes[1].axhline(tvar_pdr, color='steelblue', linestyle=':', alpha=0.6)
axes[1].axhline(aodv_pdr, color='coral', linestyle=':', alpha=0.6)
axes[1].set_xlabel('Time Window', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Cumulative PDR (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Cumulative Packet Delivery Rate', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: TVAR prediction accuracy
axes[2].scatter(tvar_results['predicted_loss'], tvar_results['actual_loss'], 
                alpha=0.6, s=50, color='steelblue', edgecolors='black', linewidth=0.5)
axes[2].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Prediction')
axes[2].set_xlabel('TVAR Predicted Loss', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Actual Loss', fontsize=11, fontweight='bold')
axes[2].set_title('TVAR Prediction Accuracy', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([-0.05, 1.05])
axes[2].set_ylim([-0.05, 1.05])

# Add text box with key result
textstr = f'PDR Improvement: +{pdr_improvement:.2f}%\nRelative Gain: {relative_improvement:+.1f}%'
props = dict(boxstyle='round', facecolor='lightgreen', alpha=0.8)
axes[2].text(0.05, 0.95, textstr, transform=axes[2].transAxes, fontsize=11,
             verticalalignment='top', bbox=props, fontweight='bold')

plt.tight_layout()
plt.savefig('09_tvar_routing_performance.png', dpi=300, bbox_inches='tight')
print("\n✓ Saved: 09_tvar_routing_performance.png")
plt.show()

print("\n" + "="*60)
print("✅ APPROACH 1 COMPLETE - SAFETY NET LOCKED")
print("="*60)
print("\nNext: Share PDR improvement with partner for dashboard")
print(f"Key result: TVAR routing achieves {pdr_improvement:+.2f}% PDR improvement")